In [1]:
import json
import random
import argparse
import time
import shutil
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional, Tuple


# ============================================================
# Defaults
# ============================================================

DEFAULT_OUTPUT = "synthetic/timebound_long.jsonl"
DEFAULT_N_EXAMPLES = 1000
DEFAULT_SEED = 42

TASK_TYPES = [
    "elapsed_time_reasoning",
    "aging_fact",
    "delayed_observation",
    "rescheduling",
    "cancellation",
    "time_window_retrieval",
    "periodic_event",
    "conflicting_updates",
]

DIFFICULTIES = ["easy", "medium", "hard"]

DOMAINS = [
    "software deployment",
    "hotel booking",
    "flight schedule",
    "warehouse access",
    "project meeting",
    "online course",
    "subscription",
    "backup job",
    "maintenance window",
    "delivery order",
    "smart-home device update",
    "class schedule",
]

ENTITIES = {
    "software deployment": [
        "deployment A",
        "deployment B",
        "version 3.5 rollout",
        "backend release",
        "mobile app release",
    ],
    "hotel booking": [
        "hotel booking",
        "conference stay",
        "city hotel reservation",
        "airport hotel stay",
    ],
    "flight schedule": [
        "flight AA123",
        "flight BA778",
        "flight QR204",
        "return flight",
    ],
    "warehouse access": [
        "warehouse code WH123",
        "dock access code",
        "gate pass",
        "temporary loading permit",
    ],
    "project meeting": [
        "milestone meeting",
        "design review",
        "performance review",
        "client meeting",
    ],
    "online course": [
        "math class",
        "online course session",
        "statistics seminar",
        "language class",
    ],
    "subscription": [
        "cloud subscription",
        "software license",
        "analytics plan",
        "support contract",
    ],
    "backup job": [
        "database backup",
        "nightly backup",
        "weekly backup",
        "archive job",
    ],
    "maintenance window": [
        "maintenance window",
        "network maintenance",
        "server patch window",
        "database maintenance",
    ],
    "delivery order": [
        "order #1234",
        "parcel delivery",
        "shipment",
        "equipment delivery",
    ],
    "smart-home device update": [
        "thermostat update",
        "smart lock update",
        "sensor firmware update",
        "lighting automation update",
    ],
    "class schedule": [
        "Math class",
        "Physics class",
        "Lab session",
        "tutorial",
    ],
}


# ============================================================
# Safety helpers
# ============================================================

def ensure_output_path(path: Path, overwrite: bool = False) -> Path:
    """
    If file exists and overwrite=False, create a timestamped alternative.
    This prevents accidental deletion of existing data.
    """
    path.parent.mkdir(parents=True, exist_ok=True)

    if not path.exists() or overwrite:
        if path.exists() and overwrite:
            backup_dir = path.parent / "backups"
            backup_dir.mkdir(exist_ok=True)
            ts = time.strftime("%Y%m%d_%H%M%S")
            backup = backup_dir / f"{path.stem}_backup_{ts}{path.suffix}"
            shutil.copy2(path, backup)
            print(f"Backup created before overwrite: {backup}")
        return path

    ts = time.strftime("%Y%m%d_%H%M%S")
    new_path = path.parent / f"{path.stem}_{ts}{path.suffix}"
    print(f"Output already exists: {path}")
    print(f"Will write to: {new_path}")
    return new_path


# ============================================================
# Time helpers
# ============================================================

def iso(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%dT%H:%M:%S")


def date_iso(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%d")


def rand_dt(base: datetime, min_days: int = 0, max_days: int = 60, hour_min: int = 7, hour_max: int = 20) -> datetime:
    d = random.randint(min_days, max_days)
    h = random.randint(hour_min, hour_max)
    m = random.choice([0, 15, 30, 45])
    return base + timedelta(days=d, hours=h, minutes=m)


def add_hours(dt: datetime, hours: int) -> datetime:
    return dt + timedelta(hours=hours)


def add_days(dt: datetime, days: int) -> datetime:
    return dt + timedelta(days=days)


def choose(xs):
    return random.choice(xs)


def difficulty_for_turns(n_turns: int) -> str:
    if n_turns < 28:
        return "easy"
    if n_turns < 40:
        return "medium"
    return "hard"


# ============================================================
# Event representation before final turn assignment
# ============================================================

def make_event(
    speaker: str,
    observation_time: datetime,
    text: str,
    event_time: Optional[datetime] = None,
    valid_from: Optional[datetime] = None,
    valid_to: Optional[datetime] = None,
    status: str = "active",
    gold: bool = False,
    tag: str = "",
) -> Dict[str, Any]:
    return {
        "_gold": gold,
        "_tag": tag,
        "speaker": speaker,
        "observation_time": iso(observation_time),
        "event_time": iso(event_time) if event_time else None,
        "valid_from": iso(valid_from) if valid_from else None,
        "valid_to": iso(valid_to) if valid_to else None,
        "status": status,
        "text": text,
    }


def finalize_example(
    ex_id: str,
    task_type: str,
    difficulty: str,
    current_time: datetime,
    events: List[Dict[str, Any]],
    query: str,
    gold_answer: str,
    required_temporal_operation: str,
    temporal_trap: str,
    explanation: str,
) -> Dict[str, Any]:
    """
    Sort by observation_time, assign final turn numbers, and collect gold turns.
    """
    events = sorted(events, key=lambda x: x["observation_time"])

    history = []
    gold_turns = []

    for idx, ev in enumerate(events):
        out_ev = {
            "turn": idx,
            "speaker": ev["speaker"],
            "observation_time": ev["observation_time"],
            "event_time": ev["event_time"],
            "valid_from": ev["valid_from"],
            "valid_to": ev["valid_to"],
            "status": ev["status"],
            "text": ev["text"],
        }

        history.append(out_ev)

        if ev.get("_gold", False):
            gold_turns.append(idx)

    return {
        "id": ex_id,
        "task_type": task_type,
        "difficulty": difficulty,
        "current_time": iso(current_time),
        "history": history,
        "query": query,
        "gold_answer": gold_answer,
        "gold_evidence_turns": gold_turns,
        "required_temporal_operation": required_temporal_operation,
        "temporal_trap": temporal_trap,
        "explanation": explanation,
    }


# ============================================================
# Distractors
# ============================================================

def random_domain_entity() -> Tuple[str, str]:
    domain = choose(DOMAINS)
    entity = choose(ENTITIES[domain])
    return domain, entity


def make_generic_distractor(base: datetime, idx: int, target_domain: str, target_entity: str) -> Dict[str, Any]:
    """
    Distractors are intentionally similar:
    - same domain but wrong entity/date;
    - same entity but old/superseded;
    - same date but different object.
    """
    mode = random.choice([
        "same_domain_wrong_entity",
        "same_entity_old",
        "same_entity_future",
        "same_date_wrong_entity",
        "cancelled_lookalike",
        "delayed_lookalike",
        "expired_code_like",
    ])

    obs = rand_dt(base, 0, 80)
    domain = target_domain
    entity = target_entity

    if mode == "same_domain_wrong_entity":
        candidates = [x for x in ENTITIES.get(domain, [entity]) if x != entity]
        wrong_entity = choose(candidates) if candidates else entity + " copy"
        event_time = add_days(obs, random.randint(1, 20))
        return make_event(
            speaker="user",
            observation_time=obs,
            event_time=event_time,
            valid_from=obs,
            valid_to=event_time,
            status="scheduled",
            text=f"{wrong_entity} is scheduled for {iso(event_time)}.",
            gold=False,
            tag="distractor_same_domain_wrong_entity",
        )

    if mode == "same_entity_old":
        old_time = add_days(obs, random.randint(1, 12))
        update_time = add_hours(obs, random.randint(2, 18))
        return make_event(
            speaker="user",
            observation_time=obs,
            event_time=old_time,
            valid_from=obs,
            valid_to=update_time,
            status="superseded",
            text=f"Earlier note: {entity} was planned for {iso(old_time)}, but this note may be outdated.",
            gold=False,
            tag="distractor_same_entity_old",
        )

    if mode == "same_entity_future":
        future_time = add_days(obs, random.randint(20, 70))
        return make_event(
            speaker="user",
            observation_time=obs,
            event_time=future_time,
            valid_from=obs,
            valid_to=None,
            status="scheduled",
            text=f"A later unrelated instance of {entity} is scheduled for {iso(future_time)}.",
            gold=False,
            tag="distractor_same_entity_future",
        )

    if mode == "same_date_wrong_entity":
        wrong_domain, wrong_entity = random_domain_entity()
        event_time = add_days(obs, random.randint(1, 15)).replace(hour=random.choice([8, 9, 10, 14, 16]), minute=0)
        return make_event(
            speaker="system",
            observation_time=obs,
            event_time=event_time,
            valid_from=obs,
            valid_to=event_time,
            status="scheduled",
            text=f"{wrong_entity} also has an event on {date_iso(event_time)} at {event_time.strftime('%H:%M')}.",
            gold=False,
            tag="distractor_same_date_wrong_entity",
        )

    if mode == "cancelled_lookalike":
        event_time = add_days(obs, random.randint(1, 18))
        cancel_time = add_hours(obs, random.randint(1, 12))
        return make_event(
            speaker="user",
            observation_time=cancel_time,
            event_time=event_time,
            valid_from=cancel_time,
            valid_to=None,
            status="cancelled",
            text=f"A similar booking for {entity} on {iso(event_time)} was cancelled.",
            gold=False,
            tag="distractor_cancelled_lookalike",
        )

    if mode == "delayed_lookalike":
        event_time = add_days(obs, -random.randint(1, 5))
        return make_event(
            speaker="system",
            observation_time=obs,
            event_time=event_time,
            valid_from=obs,
            valid_to=None,
            status="delayed",
            text=f"Delayed notification: {entity} actually happened on {iso(event_time)}, but the notice arrived at {iso(obs)}.",
            gold=False,
            tag="distractor_delayed_lookalike",
        )

    # expired_code_like
    start = obs
    end = add_days(obs, random.randint(1, 6))
    return make_event(
        speaker="user",
        observation_time=obs,
        event_time=start,
        valid_from=start,
        valid_to=end,
        status="expired",
        text=f"Temporary access for {entity} was valid from {date_iso(start)} to {date_iso(end)} only.",
        gold=False,
        tag="distractor_expired_code_like",
    )


def add_distractors(
    events: List[Dict[str, Any]],
    base: datetime,
    target_domain: str,
    target_entity: str,
    n_distractors: int,
) -> None:
    for i in range(n_distractors):
        events.append(make_generic_distractor(base, i, target_domain, target_entity))


# ============================================================
# Scenario generators
# ============================================================

def gen_rescheduling(ex_id: str, base: datetime, n_turns: int) -> Dict[str, Any]:
    domain = choose(["project meeting", "maintenance window", "hotel booking", "online course", "flight schedule"])
    entity = choose(ENTITIES[domain])

    obs0 = rand_dt(base, 0, 20)
    t0 = add_days(obs0, random.randint(7, 20)).replace(hour=random.choice([9, 10, 14, 16]), minute=0)
    obs1 = add_hours(obs0, random.randint(3, 24))
    t1 = add_days(t0, random.choice([-1, 1, 2])).replace(hour=random.choice([10, 11, 15, 17]), minute=0)
    obs2 = add_hours(obs1, random.randint(3, 48))
    t2 = add_days(t1, random.choice([-2, -1, 1, 2])).replace(hour=random.choice([8, 10, 14, 18]), minute=0)

    current_time = add_hours(obs2, random.randint(1, 12))

    events = [
        make_event(
            "user",
            obs0,
            f"{entity} is initially scheduled for {iso(t0)}.",
            event_time=t0,
            valid_from=obs0,
            valid_to=obs1,
            status="superseded",
            gold=True,
            tag="gold_old_schedule",
        ),
        make_event(
            "user",
            obs1,
            f"{entity} was rescheduled to {iso(t1)}.",
            event_time=t1,
            valid_from=obs1,
            valid_to=obs2,
            status="superseded",
            gold=True,
            tag="gold_middle_schedule",
        ),
        make_event(
            "user",
            obs2,
            f"Final update: {entity} is now scheduled for {iso(t2)}.",
            event_time=t2,
            valid_from=obs2,
            valid_to=None,
            status="active",
            gold=True,
            tag="gold_final_schedule",
        ),
    ]

    add_distractors(events, base, domain, entity, max(0, n_turns - len(events)))

    query = f"What is the current scheduled time for {entity}?"
    gold_answer = iso(t2)

    return finalize_example(
        ex_id=ex_id,
        task_type="rescheduling",
        difficulty=difficulty_for_turns(n_turns),
        current_time=current_time,
        events=events,
        query=query,
        gold_answer=gold_answer,
        required_temporal_operation="use the latest active rescheduling update and ignore superseded times",
        temporal_trap="The history contains older schedules and similar distractor events that look relevant.",
        explanation=f"The final active update schedules {entity} for {iso(t2)}.",
    )


def gen_cancellation(ex_id: str, base: datetime, n_turns: int) -> Dict[str, Any]:
    domain = choose(["project meeting", "smart-home device update", "maintenance window", "class schedule"])
    entity = choose(ENTITIES[domain])

    obs0 = rand_dt(base, 0, 20)
    event_time = add_days(obs0, random.randint(5, 18)).replace(hour=random.choice([9, 12, 15, 18]), minute=0)
    cancel_time = add_hours(obs0, random.randint(6, 72))
    current_time = add_hours(cancel_time, random.randint(1, 48))

    events = [
        make_event(
            "user",
            obs0,
            f"{entity} is scheduled for {iso(event_time)}.",
            event_time=event_time,
            valid_from=obs0,
            valid_to=cancel_time,
            status="cancelled",
            gold=True,
            tag="gold_original_scheduled",
        ),
        make_event(
            "user",
            cancel_time,
            f"Cancel {entity}; it should no longer happen at {iso(event_time)}.",
            event_time=event_time,
            valid_from=cancel_time,
            valid_to=None,
            status="cancelled",
            gold=True,
            tag="gold_cancellation",
        ),
    ]

    add_distractors(events, base, domain, entity, max(0, n_turns - len(events)))

    query = f"Is {entity} still scheduled as of {iso(current_time)}?"
    gold_answer = "No, it is cancelled."

    return finalize_example(
        ex_id=ex_id,
        task_type="cancellation",
        difficulty=difficulty_for_turns(n_turns),
        current_time=current_time,
        events=events,
        query=query,
        gold_answer=gold_answer,
        required_temporal_operation="apply the cancellation update instead of the original scheduled event",
        temporal_trap="The original schedule remains in memory and is lexically similar to the query.",
        explanation=f"{entity} was cancelled at {iso(cancel_time)}.",
    )


def gen_aging_fact(ex_id: str, base: datetime, n_turns: int) -> Dict[str, Any]:
    domain = choose(["warehouse access", "subscription", "software deployment"])
    entity = choose(ENTITIES[domain])

    start = rand_dt(base, 0, 20)
    duration_days = random.choice([2, 3, 5, 7, 10])
    end = add_days(start, duration_days)
    query_time = add_days(end, random.choice([-1, 0, 1, 3, 7]))
    still_valid = query_time <= end

    events = [
        make_event(
            "user",
            start,
            f"{entity} is valid from {iso(start)} until {iso(end)}.",
            event_time=start,
            valid_from=start,
            valid_to=end,
            status="active" if still_valid else "expired",
            gold=True,
            tag="gold_validity",
        )
    ]

    # Optional renewal in some cases
    if random.random() < 0.45:
        renewal_obs = add_days(start, random.randint(1, max(1, duration_days - 1)))
        new_end = add_days(end, random.randint(3, 8))
        still_valid = query_time <= new_end
        events[0]["status"] = "superseded"
        events[0]["valid_to"] = iso(renewal_obs)

        events.append(
            make_event(
                "user",
                renewal_obs,
                f"Renewal update: {entity} validity is extended until {iso(new_end)}.",
                event_time=renewal_obs,
                valid_from=renewal_obs,
                valid_to=new_end,
                status="active" if still_valid else "expired",
                gold=True,
                tag="gold_renewal",
            )
        )
        final_end = new_end
    else:
        final_end = end

    answer_yes = query_time <= final_end
    gold_answer = "Yes" if answer_yes else f"No, it expired on {iso(final_end)}."

    add_distractors(events, base, domain, entity, max(0, n_turns - len(events)))

    query = f"Is {entity} valid on {date_iso(query_time)}?"

    return finalize_example(
        ex_id=ex_id,
        task_type="aging_fact",
        difficulty=difficulty_for_turns(n_turns),
        current_time=query_time,
        events=events,
        query=query,
        gold_answer=gold_answer,
        required_temporal_operation="check the validity interval against the queried date",
        temporal_trap="The memory contains old validity periods and similar expired access facts.",
        explanation=f"The final validity end is {iso(final_end)}, and the queried time is {iso(query_time)}.",
    )


def gen_delayed_observation(ex_id: str, base: datetime, n_turns: int) -> Dict[str, Any]:
    domain = choose(["delivery order", "flight schedule", "software deployment"])
    entity = choose(ENTITIES[domain])

    event_time = rand_dt(base, 5, 30)
    observation_time = add_hours(event_time, random.choice([6, 12, 24, 48, 72]))
    current_time = add_hours(observation_time, random.randint(1, 12))

    ask_actual = random.random() < 0.55

    events = [
        make_event(
            "system",
            observation_time,
            f"Delayed notification: {entity} actually completed at {iso(event_time)}, but the notice arrived at {iso(observation_time)}.",
            event_time=event_time,
            valid_from=observation_time,
            valid_to=None,
            status="delayed",
            gold=True,
            tag="gold_delayed_observation",
        )
    ]

    add_distractors(events, base, domain, entity, max(0, n_turns - len(events)))

    if ask_actual:
        query = f"When exactly did {entity} actually happen?"
        gold_answer = iso(event_time)
        op = "distinguish event_time from observation_time"
        trap = "A naive model may answer with the notification time instead of the actual event time."
    else:
        query_date = date_iso(event_time + timedelta(days=random.choice([-1, 0, 1])))
        query = f"Did {entity} happen on {query_date}?"
        gold_answer = "Yes" if query_date == date_iso(event_time) else "No"
        op = "compare the event_time date with the queried date"
        trap = "The model may confuse the date of notification with the date of the event."

    return finalize_example(
        ex_id=ex_id,
        task_type="delayed_observation",
        difficulty=difficulty_for_turns(n_turns),
        current_time=current_time,
        events=events,
        query=query,
        gold_answer=gold_answer,
        required_temporal_operation=op,
        temporal_trap=trap,
        explanation=f"The actual event time is {iso(event_time)}, while the observation time is {iso(observation_time)}.",
    )


def gen_time_window_retrieval(ex_id: str, base: datetime, n_turns: int) -> Dict[str, Any]:
    domain = choose(["hotel booking", "subscription", "warehouse access", "project meeting"])
    entity = choose(ENTITIES[domain])

    t1 = rand_dt(base, 0, 20)
    t2 = add_days(t1, random.randint(4, 9))
    t3 = add_days(t2, random.randint(4, 9))

    query_time = t2 + timedelta(days=random.randint(1, 3), hours=12)
    current_time = add_days(t3, random.randint(1, 7))

    state_a = random.choice(["Office A", "basic plan", "temporary access", "hotel stay"])
    state_b = random.choice(["Office B", "premium plan", "renewed access", "extended hotel stay"])

    events = [
        make_event(
            "user",
            t1,
            f"For {entity}, the active state is {state_a} from {iso(t1)} until {iso(t2)}.",
            event_time=t1,
            valid_from=t1,
            valid_to=t2,
            status="historical",
            gold=False,
            tag="old_state",
        ),
        make_event(
            "user",
            t2,
            f"Update for {entity}: the active state is now {state_b} from {iso(t2)} until {iso(t3)}.",
            event_time=t2,
            valid_from=t2,
            valid_to=t3,
            status="historical",
            gold=True,
            tag="gold_state_at_query",
        ),
        make_event(
            "user",
            t3,
            f"Later update for {entity}: the previous state ended at {iso(t3)}.",
            event_time=t3,
            valid_from=t3,
            valid_to=None,
            status="active",
            gold=True,
            tag="gold_later_boundary",
        ),
    ]

    add_distractors(events, base, domain, entity, max(0, n_turns - len(events)))

    query = f"What was the active state for {entity} on {iso(query_time)}?"
    gold_answer = state_b

    return finalize_example(
        ex_id=ex_id,
        task_type="time_window_retrieval",
        difficulty=difficulty_for_turns(n_turns),
        current_time=current_time,
        events=events,
        query=query,
        gold_answer=gold_answer,
        required_temporal_operation="retrieve the memory valid at the queried past time window",
        temporal_trap="The latest current state is not necessarily the state valid at the queried time.",
        explanation=f"At {iso(query_time)}, {entity} was in state {state_b}.",
    )


def next_weekday_after(dt: datetime, weekday: int) -> datetime:
    """
    weekday: Monday=0 ... Sunday=6
    Strictly after dt.
    """
    days = (weekday - dt.weekday()) % 7
    if days == 0:
        days = 7
    return (dt + timedelta(days=days)).replace(hour=0, minute=0, second=0)


def gen_periodic_event(ex_id: str, base: datetime, n_turns: int) -> Dict[str, Any]:
    domain = choose(["backup job", "class schedule", "subscription", "software deployment"])
    entity = choose(ENTITIES[domain])

    start = rand_dt(base, 0, 15).replace(hour=8, minute=0)
    weekday = random.choice([0, 2, 4])  # Mon/Wed/Fri
    event_hour = random.choice([8, 10, 14, 18])
    end = add_days(start, random.choice([35, 49, 63, 84]))

    update_obs = add_days(start, random.randint(7, 20))
    new_hour = random.choice([9, 11, 15, 19])
    query_after = add_days(update_obs, random.randint(5, 25))
    next_date = next_weekday_after(query_after, weekday).replace(hour=new_hour, minute=0)

    if next_date > end:
        gold_answer = "No future occurrence before the recurrence ends."
    else:
        gold_answer = iso(next_date)

    current_time = query_after

    weekday_name = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"][weekday]

    events = [
        make_event(
            "user",
            start,
            f"{entity} occurs every {weekday_name} at {event_hour:02d}:00 from {date_iso(start)} until {date_iso(end)}.",
            event_time=start,
            valid_from=start,
            valid_to=end,
            status="active",
            gold=True,
            tag="gold_recurrence_rule",
        ),
        make_event(
            "user",
            update_obs,
            f"Schedule update: starting {date_iso(update_obs)}, {entity} occurs at {new_hour:02d}:00 instead of {event_hour:02d}:00.",
            event_time=update_obs,
            valid_from=update_obs,
            valid_to=end,
            status="active",
            gold=True,
            tag="gold_recurrence_update",
        ),
    ]

    # Add one cancelled occurrence distractor that should not affect the queried next date unless same date
    cancel_obs = add_days(update_obs, random.randint(1, 10))
    cancelled_date = next_weekday_after(cancel_obs, weekday).replace(hour=new_hour, minute=0)
    if cancelled_date != next_date:
        events.append(
            make_event(
                "user",
                cancel_obs,
                f"One occurrence of {entity} on {iso(cancelled_date)} was cancelled only for that date.",
                event_time=cancelled_date,
                valid_from=cancel_obs,
                valid_to=cancelled_date,
                status="cancelled",
                gold=False,
                tag="distractor_cancelled_occurrence",
            )
        )

    add_distractors(events, base, domain, entity, max(0, n_turns - len(events)))

    query = f"What is the next occurrence of {entity} after {iso(query_after)}?"

    return finalize_example(
        ex_id=ex_id,
        task_type="periodic_event",
        difficulty=difficulty_for_turns(n_turns),
        current_time=current_time,
        events=events,
        query=query,
        gold_answer=gold_answer,
        required_temporal_operation="combine recurrence rule with later time update and recurrence end date",
        temporal_trap="The history contains old recurrence times, cancelled one-off occurrences, and similar periodic events.",
        explanation=f"The relevant rule is every {weekday_name}, updated to {new_hour:02d}:00.",
    )


def gen_conflicting_updates(ex_id: str, base: datetime, n_turns: int) -> Dict[str, Any]:
    domain = choose(["software deployment", "delivery order", "flight schedule", "subscription"])
    entity = choose(ENTITIES[domain])

    obs0 = rand_dt(base, 0, 20)
    t0 = add_days(obs0, random.randint(5, 15)).replace(hour=10, minute=0)

    obs1 = add_hours(obs0, random.randint(4, 24))
    t1 = add_days(t0, random.choice([-1, 1, 2])).replace(hour=11, minute=0)

    obs2 = add_hours(obs1, random.randint(4, 24))
    t2 = add_days(t1, random.choice([-2, -1, 1, 2])).replace(hour=9, minute=0)

    current_time = add_hours(obs2, random.randint(2, 24))

    events = [
        make_event(
            "user",
            obs0,
            f"Initial update: {entity} is scheduled for {iso(t0)}.",
            event_time=t0,
            valid_from=obs0,
            valid_to=obs1,
            status="superseded",
            gold=True,
            tag="gold_conflict_1",
        ),
        make_event(
            "user",
            obs1,
            f"Second update: {entity} is now scheduled for {iso(t1)}.",
            event_time=t1,
            valid_from=obs1,
            valid_to=obs2,
            status="superseded",
            gold=True,
            tag="gold_conflict_2",
        ),
        make_event(
            "user",
            obs2,
            f"Confirmed final update: {entity} is scheduled for {iso(t2)}.",
            event_time=t2,
            valid_from=obs2,
            valid_to=None,
            status="active",
            gold=True,
            tag="gold_conflict_final",
        ),
    ]

    add_distractors(events, base, domain, entity, max(0, n_turns - len(events)))

    query = f"What is the latest confirmed scheduled time for {entity}?"
    gold_answer = iso(t2)

    return finalize_example(
        ex_id=ex_id,
        task_type="conflicting_updates",
        difficulty=difficulty_for_turns(n_turns),
        current_time=current_time,
        events=events,
        query=query,
        gold_answer=gold_answer,
        required_temporal_operation="resolve conflicting updates using the latest active update",
        temporal_trap="Earlier conflicting updates are lexically similar and may be retrieved by semantic similarity.",
        explanation=f"The final confirmed update schedules {entity} for {iso(t2)}.",
    )


def gen_elapsed_time_reasoning(ex_id: str, base: datetime, n_turns: int) -> Dict[str, Any]:
    domain = choose(["software deployment", "online course", "subscription", "backup job"])
    entity = choose(ENTITIES[domain])

    start = rand_dt(base, 0, 20).replace(minute=0)
    pause = add_days(start, random.randint(2, 5)).replace(hour=random.choice([9, 12, 18]), minute=0)
    resume = add_days(pause, random.randint(1, 3)).replace(hour=random.choice([8, 10, 14]), minute=0)
    query_time = add_days(resume, random.randint(1, 4)).replace(hour=random.choice([10, 15, 20]), minute=0)

    active_1 = int((pause - start).total_seconds() // 3600)
    active_2 = int((query_time - resume).total_seconds() // 3600)
    total_hours = active_1 + active_2

    events = [
        make_event(
            "user",
            start,
            f"{entity} became active at {iso(start)}.",
            event_time=start,
            valid_from=start,
            valid_to=pause,
            status="historical",
            gold=True,
            tag="gold_start",
        ),
        make_event(
            "user",
            pause,
            f"{entity} was paused at {iso(pause)}.",
            event_time=pause,
            valid_from=pause,
            valid_to=resume,
            status="historical",
            gold=True,
            tag="gold_pause",
        ),
        make_event(
            "user",
            resume,
            f"{entity} resumed at {iso(resume)}.",
            event_time=resume,
            valid_from=resume,
            valid_to=None,
            status="active",
            gold=True,
            tag="gold_resume",
        ),
    ]

    add_distractors(events, base, domain, entity, max(0, n_turns - len(events)))

    query = f"Calculate the total active time for {entity} until {iso(query_time)}."
    gold_answer = f"{total_hours} hours"

    return finalize_example(
        ex_id=ex_id,
        task_type="elapsed_time_reasoning",
        difficulty=difficulty_for_turns(n_turns),
        current_time=query_time,
        events=events,
        query=query,
        gold_answer=gold_answer,
        required_temporal_operation="sum active intervals while excluding paused time",
        temporal_trap="The model may count calendar time from start to query time and ignore the pause interval.",
        explanation=f"Active intervals are {active_1} hours and {active_2} hours, totaling {total_hours} hours.",
    )


GEN_FUNCS = {
    "elapsed_time_reasoning": gen_elapsed_time_reasoning,
    "aging_fact": gen_aging_fact,
    "delayed_observation": gen_delayed_observation,
    "rescheduling": gen_rescheduling,
    "cancellation": gen_cancellation,
    "time_window_retrieval": gen_time_window_retrieval,
    "periodic_event": gen_periodic_event,
    "conflicting_updates": gen_conflicting_updates,
}


# ============================================================
# Validation and stats
# ============================================================

def validate_example(ex: Dict[str, Any]) -> List[str]:
    errors = []

    required = [
        "id",
        "task_type",
        "difficulty",
        "current_time",
        "history",
        "query",
        "gold_answer",
        "gold_evidence_turns",
        "required_temporal_operation",
        "temporal_trap",
        "explanation",
    ]

    for k in required:
        if k not in ex:
            errors.append(f"missing field: {k}")

    if "history" in ex:
        if not isinstance(ex["history"], list):
            errors.append("history is not a list")
        else:
            turns = [ev.get("turn") for ev in ex["history"]]
            if turns != list(range(len(ex["history"]))):
                errors.append("turns are not sequential from 0")

            for ev in ex["history"]:
                for k in [
                    "turn",
                    "speaker",
                    "observation_time",
                    "event_time",
                    "valid_from",
                    "valid_to",
                    "status",
                    "text",
                ]:
                    if k not in ev:
                        errors.append(f"event missing field: {k}")

    if "gold_evidence_turns" in ex and "history" in ex:
        n = len(ex["history"])
        for t in ex["gold_evidence_turns"]:
            if not isinstance(t, int) or t < 0 or t >= n:
                errors.append(f"bad gold evidence turn: {t}")

    if not ex.get("gold_evidence_turns"):
        errors.append("empty gold_evidence_turns")

    if len(ex.get("history", [])) < 20:
        errors.append("history too short for TimeBound-Long")

    return errors


def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def make_splits(rows: List[Dict[str, Any]], output_path: Path) -> None:
    split_dir = output_path.parent / f"{output_path.stem}_splits"
    split_dir.mkdir(parents=True, exist_ok=True)

    shuffled = list(rows)
    random.shuffle(shuffled)

    n = len(shuffled)
    n_train = int(0.70 * n)
    n_dev = int(0.10 * n)

    splits = {
        "train": shuffled[:n_train],
        "dev": shuffled[n_train:n_train + n_dev],
        "test": shuffled[n_train + n_dev:],
    }

    for name, data in splits.items():
        write_jsonl(split_dir / f"{name}.jsonl", data)
        print(f"Split {name}: {len(data)} -> {split_dir / f'{name}.jsonl'}")


def summarize(rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    task_counts = {}
    diff_counts = {}
    turn_counts = []
    gold_counts = []

    for r in rows:
        task_counts[r["task_type"]] = task_counts.get(r["task_type"], 0) + 1
        diff_counts[r["difficulty"]] = diff_counts.get(r["difficulty"], 0) + 1
        turn_counts.append(len(r["history"]))
        gold_counts.append(len(r["gold_evidence_turns"]))

    return {
        "n_examples": len(rows),
        "task_counts": dict(sorted(task_counts.items())),
        "difficulty_counts": dict(sorted(diff_counts.items())),
        "turns_min": min(turn_counts) if turn_counts else None,
        "turns_max": max(turn_counts) if turn_counts else None,
        "turns_avg": sum(turn_counts) / len(turn_counts) if turn_counts else None,
        "gold_turns_min": min(gold_counts) if gold_counts else None,
        "gold_turns_max": max(gold_counts) if gold_counts else None,
        "gold_turns_avg": sum(gold_counts) / len(gold_counts) if gold_counts else None,
    }


# ============================================================
# Main generation
# ============================================================

def generate_dataset(
    n_examples: int,
    seed: int,
    turns_min: int,
    turns_max: int,
) -> List[Dict[str, Any]]:
    random.seed(seed)

    base = datetime(2026, 1, 1, 0, 0, 0)
    rows = []

    for i in range(n_examples):
        task_type = TASK_TYPES[i % len(TASK_TYPES)]
        gen = GEN_FUNCS[task_type]

        n_turns = random.randint(turns_min, turns_max)
        ex_id = f"timebound_long_{i:06d}"

        ex = gen(ex_id, base, n_turns)
        rows.append(ex)

    random.shuffle(rows)

    # Reassign stable IDs after shuffle
    for i, ex in enumerate(rows):
        ex["id"] = f"timebound_long_{i:06d}"

    return rows


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--output", type=str, default=DEFAULT_OUTPUT)
    parser.add_argument("--n_examples", type=int, default=DEFAULT_N_EXAMPLES)
    parser.add_argument("--seed", type=int, default=DEFAULT_SEED)

    parser.add_argument("--turns_min", type=int, default=20)
    parser.add_argument("--turns_max", type=int, default=50)

    parser.add_argument("--overwrite", action="store_true")
    parser.add_argument("--make_splits", action="store_true")

    # Jupyter-safe
    args, unknown = parser.parse_known_args()
    if unknown:
        print("Ignored unknown arguments:", unknown)

    output_path = ensure_output_path(Path(args.output), overwrite=args.overwrite)

    print("=" * 100)
    print("Generate TimeBound-Long")
    print("=" * 100)
    print(f"Output: {output_path}")
    print(f"N examples: {args.n_examples}")
    print(f"Seed: {args.seed}")
    print(f"Turns per example: {args.turns_min}–{args.turns_max}")
    print("=" * 100)

    rows = generate_dataset(
        n_examples=args.n_examples,
        seed=args.seed,
        turns_min=args.turns_min,
        turns_max=args.turns_max,
    )

    print("Validating...")
    bad = []

    for ex in rows:
        errors = validate_example(ex)
        if errors:
            bad.append((ex["id"], errors))

    if bad:
        print(f"Validation found {len(bad)} bad examples.")
        for ex_id, errors in bad[:20]:
            print(ex_id, errors)
        raise RuntimeError("Validation failed. Fix generator before saving.")

    write_jsonl(output_path, rows)

    summary = summarize(rows)
    summary_path = output_path.with_suffix(".summary.json")

    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\nSaved:")
    print(f"  dataset: {output_path}")
    print(f"  summary: {summary_path}")

    print("\nSummary:")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    if args.make_splits:
        make_splits(rows, output_path)

    print("\nDone.")


if __name__ == "__main__":
    main()

Ignored unknown arguments: ['-f', 'C:\\Users\\ivan\\AppData\\Roaming\\jupyter\\runtime\\kernel-e1d70910-7124-4fc0-a5f4-b4fee35b0b8b.json']
Generate TimeBound-Long
Output: synthetic\timebound_long.jsonl
N examples: 1000
Seed: 42
Turns per example: 20–50
Validating...

Saved:
  dataset: synthetic\timebound_long.jsonl
  summary: synthetic\timebound_long.summary.json

Summary:
{
  "n_examples": 1000,
  "task_counts": {
    "aging_fact": 125,
    "cancellation": 125,
    "conflicting_updates": 125,
    "delayed_observation": 125,
    "elapsed_time_reasoning": 125,
    "periodic_event": 125,
    "rescheduling": 125,
    "time_window_retrieval": 125
  },
  "difficulty_counts": {
    "easy": 241,
    "hard": 349,
    "medium": 410
  },
  "turns_min": 20,
  "turns_max": 50,
  "turns_avg": 34.98,
  "gold_turns_min": 1,
  "gold_turns_max": 3,
  "gold_turns_avg": 2.178
}

Done.
